In [34]:
import igl
from pathlib import Path
import csv
import math
import numpy as np

# Use absolute path
base_path = Path("/Users/aa13586/Desktop/symmetric-dirichlet")

uv_path = base_path / "data" / "closed-Myles-dijkstra-01-11" / "filigree100k_param.obj"
v3d, uv, _, f, fuv, _ = igl.readOBJ(uv_path)
print(len(v3d), len(uv), len(f), len(fuv))

149942 174325 300140 300140


In [35]:
import re

def read_vertex_list(path):
    verts = []
    with open(path, newline='') as csvfile:
        reader = csv.reader(csvfile)
        for row in reader:
            if not row:
                continue
            text = " ".join(row).strip()
            if text == "":
                continue
            for tok in re.split(r'[,\s]+', text):
                if not tok:
                    continue
                try:
                    verts.append(int(tok))
                except ValueError:
                    # skip non-integer tokens
                    continue
    return verts


folder = base_path / "output"
input_indices = read_vertex_list(folder / "max_hessian_vertex.csv")
input_faces = read_vertex_list(folder / "max_hessian_faces.csv")
print(len(input_indices), len(input_faces))

88 21


In [26]:
def triangle_area(v0, v1, v2):
    a = np.linalg.norm(v1 - v0)
    b = np.linalg.norm(v2 - v1)
    c = np.linalg.norm(v0 - v2)
    s = (a + b + c) / 2.0
    area = max(s * (s - a) * (s - b) * (s - c), 1e-10)**0.5
    inradius = area / s
    circumradius = (a * b * c) / (4.0 * area)
    return area, circumradius / inradius
# build adjacency: faces incident to each vertex
faces_uv = []
for f_0 in range(len(fuv)):
    if fuv[f_0][0] in input_indices or fuv[f_0][1] in input_indices or fuv[f_0][2] in input_indices:
        faces_uv.append(f_0)
print(faces_uv[:5], len(faces_uv))

[1737, 1738, 1745, 2217, 2239] 400


In [47]:
import csv

def triangle_aspect_ratio(v0, v1, v2):
    a = np.linalg.norm(v1 - v0)
    b = np.linalg.norm(v2 - v1)
    c = np.linalg.norm(v0 - v2)
    s = (a + b + c) / 2.0
    area = (s * (s - a) * (s - b) * (s - c))**0.5
    inradius = area / s
    circumradius = (a * b * c) / (4.0 * area)
    return area, circumradius / inradius

def compute_face_metrics(uv, faces_list, f):
    """
    Compute metrics for each face in faces_uv:
    - face index
    - area (UV)
    - aspect ratio (circumradius / inradius)
    - min edge length
    - length of edge containing vertex from input_indices
    """
    rows = []
    
    for face_idx in faces_list:
        v0_idx, v1_idx, v2_idx = f[face_idx]
        v0 = uv[v0_idx]
        v1 = uv[v1_idx]
        v2 = uv[v2_idx]
        
        # Edge lengths in UV space
        e0 = np.linalg.norm(v1 - v2)  # edge opposite v0
        e1 = np.linalg.norm(v2 - v0)  # edge opposite v1
        e2 = np.linalg.norm(v0 - v1)  # edge opposite v2

        area, aspect_ratio = triangle_aspect_ratio(v0, v1, v2)

        # Min edge
        min_edge = min(e0, e1, e2)
        max_edge = max(e0, e1, e2)

        # Compute angles using law of cosines: cos(A) = (b² + c² - a²) / (2bc)
        # Angle at v0 (opposite e0), angle at v1 (opposite e1), angle at v2 (opposite e2)
        angle0 = np.arccos(np.clip((e1**2 + e2**2 - e0**2) / (2 * e1 * e2), -1, 1))
        angle1 = np.arccos(np.clip((e0**2 + e2**2 - e1**2) / (2 * e0 * e2), -1, 1))
        angle2 = np.arccos(np.clip((e0**2 + e1**2 - e2**2) / (2 * e0 * e1), -1, 1))
        
        min_angle_rad = min(angle0, angle1, angle2)
        min_angle_deg = np.degrees(min_angle_rad)
        # Find edge length containing vertex from input_indices
        edge_with_special_vertex = None
        for vi in [v0_idx, v1_idx, v2_idx]:
            if vi in input_indices:
                if vi == v0_idx:
                    # v0 is special; edges are e1 and e2
                    edge_with_special_vertex = max(e1, e2)
                elif vi == v1_idx:
                    # v1 is special; edges are e0 and e2
                    edge_with_special_vertex = max(e0, e2)
                else:  # vi == v2_idx
                    # v2 is special; edges are e0 and e1
                    edge_with_special_vertex = max(e0, e1)
                break
        
        if edge_with_special_vertex is None:
            edge_with_special_vertex = 0.0  # no special vertex in face
        
        rows.append([
            face_idx,  # tuple of indices
            area,
            aspect_ratio,
            min_edge,
            max_edge,
            min_angle_deg
        ])
    
    return rows

# Compute metrics
metrics = compute_face_metrics(uv, faces_uv, fuv)

# Write to CSV
output_csv = base_path / "output" / "UVface_metrics.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)
print(max([row[2] for row in metrics]))
with open(output_csv, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["face_indices", "area_uv", "aspect_ratio", "min_edge", "max_edge", "min_angle_deg"])
    for row in metrics:
        writer.writerow([row[0], row[1], row[2], row[3], row[4], row[5]])

print(f"Wrote {len(metrics)} face metrics to {output_csv}")

1810.8997246154056
Wrote 400 face metrics to /Users/aa13586/Desktop/symmetric-dirichlet/output/UVface_metrics.csv


In [42]:
print(len(f))

300140


In [57]:
def compute_aspect_ratio(vs, f):
    """Compute aspect ratio statistics for the mesh."""
    def triangle_aspect_ratio(v0, v1, v2):
        a = np.linalg.norm(v1 - v0)
        b = np.linalg.norm(v2 - v1)
        c = np.linalg.norm(v0 - v2)
        s = (a + b + c) / 2.0
        area = (s * (s - a) * (s - b) * (s - c))**0.5
        inradius = area / s
        circumradius = (a * b * c) / (4.0 * area)
        return circumradius / inradius

    aspect_ratios = []
    for face in f:
        v0, v1, v2 = vs[face]
        ar = triangle_aspect_ratio(v0, v1, v2)
        aspect_ratios.append(ar)

    aspect_ratios = np.array(aspect_ratios)
    return aspect_ratios

print(max(compute_aspect_ratio(uv, fuv[faces_uv])))
print(max(compute_aspect_ratio(v3d, f[input_faces])))

1810.8997246154056
69.19548570740848
